# Hexagon assignments

Assign all ETCs in all regions to h3 hexagons, which will be used to limit spatial linkage in the model train/test split.

In [1]:
import h3
import shapely
import pandas as pd
import tobler
import geopandas as gpd

In [2]:
h3_resolution = 7

In [3]:
regions_datadir = "/data/uscuni-eurofab/"
tessellations_dir = '/data/uscuni-eurofab/processed_data/tessellations/'
buildings_dir = '/data/uscuni-eurofab/processed_data/buildings/'

region_hulls = gpd.read_parquet(
        regions_datadir + "regions/" + "ms_ce_region_hulls.parquet"
    )
region_hulls.shape

(474, 1)

In [4]:
def assign_hexagons(region_id, region_hull):
    '''Assign all ETCs in a reigion to h3 hexagons.'''
    
    ## split region hull into hexagons
    bounds = region_hull.iloc[0]
    poly = h3.geo_to_cells(bounds, res=h3_resolution)
    res = [shapely.geometry.shape(h3.cells_to_geo([p])) for p in poly]
    hexagons = gpd.GeoSeries(res, index=poly,name='geometry', crs='epsg:4326').to_crs(epsg=3035)

    tess = gpd.read_parquet(
            tessellations_dir + f"tessellation_{region_id}.parquet"
    )

    # assign hexagons to tessellation cells
    inp, res = tess.sindex.query(hexagons, predicate='intersects')
    # polygons should be assigned to only one h3 grid
    duplicated = pd.Series(res).duplicated()
    inp = inp[~duplicated]
    res = res[~duplicated]
    
    hex_assignments = pd.Series(hexagons.index[inp].values, tess.index[res], name='hexagons').sort_index()
    return hex_assignments

In [ ]:
%%time
for region_id, region_hull in region_hulls.to_crs(epsg=4326).iterrows():
    print(region_id)
    hex_assignments = assign_hexagons(region_id, region_hull)
    hex_assignments.reset_index().to_parquet(f'/data/uscuni-eurofab/processed_data/hexagons/{region_id}_hexagon.pq')

19
24
33
478
754


## Explore assignment

In [ ]:
region_id = 65806
# region_id = 66292
hex_assignments = pd.read_parquet(f'/data/uscuni-eurofab/processed_data/hexagons/{region_id}_hexagon.pq').set_index('index')

In [ ]:
selected = hex_assignments[hex_assignments.hexagons == '871e354ddffffff'].index
selected.shape

(1383,)

In [ ]:
tess = gpd.read_parquet(
            tessellations_dir + f"tessellation_{region_id}.parquet"
    )

In [ ]:
tess.loc[selected].explore()